# Auto-encodeurs débruiteurs avec Keras

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Import de TensorFlow et des autres librairies nécessaires

In [ ]:
import matplotlib.pyplot as plt
import numpy
import tensorflow as tf
import keras

## Chargement de MNIST

Nous allons utiliser un prétraîtement légèrement différent des autres fois : étant donné que nous voulons pouvoir prédire les valeurs données en entrée en sortie (principe de l'auto-encodage), nous allons simplement projeter ces valeurs dans $[0, 1]$ au lieu de $[0, 255]$. Notez qu'habituellement nous ne faisons pas ça : nous normalisons en centrant sur zéro et en divisant par l'écart-type.

In [ ]:
(X_train, _), (X_test, y_test) = keras.datasets.mnist.load_data()
nb_classes = 10
input_dim = 28 * 28
X_train = X_train.reshape(-1, input_dim) / 255.0
X_test = X_test.reshape(-1, input_dim) / 255.0

## Application d'un bruit  gaussien

In [ ]:
noise_factor = 0.5
X_train_noisy = X_train + numpy.random.normal(0, noise_factor, X_train.shape)
X_test_noisy = X_test + numpy.random.normal(0, noise_factor, X_test.shape)

# On clip les valeurs pour éviter les pixels plus blanc que blanc (ou plus noir
# que noir)
numpy.clip(X_train_noisy, 0, 1, out=X_train_noisy)
numpy.clip(X_test_noisy, 0, 1, out=X_test_noisy)

In [ ]:
n = 10
f, ax = plt.subplots(1, n, figsize=(n * 1.4, 2))
for i in range(n):
    ax[i].imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
    ax[i].set_title(y_test[i])
    ax[i].axis("off")
plt.show()

## Création de l'autoencodeur débruiteur



In [ ]:
encoding_dim = 4

autoencoder = keras.Sequential(
    [keras.layers.Input(shape=(input_dim,)),
     keras.layers.Dense(encoding_dim, activation="leaky_relu"),
     keras.layers.Dense(input_dim, activation="sigmoid")],
    name="autoencoder")
autoencoder.summary()

autoencoder.compile(optimizer="adam", loss="mean_squared_error")

## Apprentissage

In [ ]:
autoencoder.fit(X_train_noisy, X_train,
                epochs=50,
                batch_size=256,
                validation_split=0.2)

## Base de Test

In [ ]:
X_test_noisy_pred = autoencoder.predict(X_test_noisy)

## Affichage visuel de la performance

In [ ]:
n = 10
_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for i, (ax_top, ax_bottom) in enumerate(ax.T):
    # L'original en haut
    ax_top.imshow(X_test_noisy[i].reshape(28, 28), cmap="gray_r")
    ax_top.set_title(str(y_test[i]))
    ax_top.axis("off")

    # La reconstruction en bas
    ax_bottom.imshow(X_test_noisy_pred[i].reshape(28, 28), cmap="gray_r")
    ax_bottom.axis("off")
plt.show()

## Utilisation des réseaux convolutifs

Pour cela il faut remettre chaque image sous forme 28x28x1. Les CNNs ont besoin de cette 3ème dimension de tenseur (il pourrait y avoir plus de canaux que le niveau de gris : il y en a 3 pour les images en couleur et plus encore dans les couches intermédiaires d'un réseau convolutif où le nombre de canaux en entrée d'une couche sera le nombre de kernels de la couche précédente).

In [ ]:
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)
X_train_noisy = X_train_noisy.reshape(-1, 28, 28, 1)
X_test_noisy = X_test_noisy.reshape(-1, 28, 28, 1)

## Création du modèle

On utilisera des séquences de `Conv2D`, `MaxPool2D` avec des kernel de respectivement 3 x 3 et 2 x 2 pour la partie encodeur. Dans ces deux layers, il faudra utiliser l'option `padding="same"` afin d'éviter les effets de bord (l'image étant déjà assez petite comme ça).

Pour la partie décodeur, on utilisera des `Conv2D` de même nature suivis par des `UpSampling2D((2, 2))`, qui correspondent à l'opération inverse de `MaxPool2D`

In [ ]:
encoding_dim = 4

autoencoder = keras.Sequential()


def conv(n, activation="leaky_relu"):
  return keras.layers.Conv2D(n, (3, 3), activation=activation, padding="same")


def max_pool():
  return keras.layers.MaxPool2D((2, 2), padding="same")


def up_sampling():
  return keras.layers.UpSampling2D((2, 2))


# Encodage
autoencoder.add(keras.layers.Input(shape=X_train.shape[1:]))
autoencoder.add(conv(32))
autoencoder.add(max_pool())
autoencoder.add(conv(32))
autoencoder.add(max_pool())
autoencoder.add(conv(1))
autoencoder.add(keras.layers.Flatten())
autoencoder.add(keras.layers.Dense(encoding_dim))

# Décodage
autoencoder.add(keras.layers.Dense(49))
autoencoder.add(keras.layers.Reshape((7, 7, 1)))
autoencoder.add(conv(32))
autoencoder.add(up_sampling())
autoencoder.add(conv(32))
autoencoder.add(up_sampling())
autoencoder.add(conv(1, "sigmoid"))

autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

autoencoder.summary()

## Apprentissage

In [ ]:
autoencoder.fit(X_train_noisy, X_train,
                epochs=100,
                batch_size=128,
                validation_split=0.2)

## Affichage des performances

In [ ]:
X_test_noisy_pred = autoencoder.predict(X_test_noisy).reshape(-1, 28, 28)

n = 10

random_indexes = numpy.random.choice(X_test.shape[0],
                                     replace=False,
                                     size=n)

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
    # L'image originale en haut
    ax_top.set_title(str(y_test[random_index]))
    ax_top.imshow(X_test_noisy[random_index].reshape(28, 28), cmap="gray_r")
    ax_top.axis("off")

    # L'image reconstruite en bas
    ax_bottom.imshow(X_test_noisy_pred[random_index], cmap="gray_r")
    ax_bottom.axis("off")
plt.show()

## Sur des images non-bruitées

In [ ]:
X_test_pred = autoencoder.predict(X_test).reshape(-1, 28, 28)

n = 10

random_indexes = numpy.random.choice(X_test.shape[0],
                                     replace=False,
                                     size=n)

_, ax = plt.subplots(2, n, figsize=(n * 1.4, 4))
for (ax_top, ax_bottom), random_index in zip(ax.T, random_indexes):
    # L'image originale en haut
    ax_top.set_title(str(y_test[random_index]))
    ax_top.imshow(X_test[random_index].reshape(28, 28), cmap="gray_r")
    ax_top.axis("off")

    # L'image reconstruite en bas
    ax_bottom.imshow(X_test_pred[random_index], cmap="gray_r")
    ax_bottom.axis("off")
plt.show()

## Auto-encodeurs variationnels

Pour les VAEs, la méthode d'apprentissage est beaucoup moins facile à mettre en place, en partie à cause de [l'astuce de re-paramétrisation](https://stats.stackexchange.com/questions/199605/how-does-the-reparameterization-trick-for-vaes-work-and-why-is-it-important)). La suggestion des mainteneurs de Keras est d'utiliser le code suivant :

In [ ]:
class Sampling(keras.layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
latent_dim = 2

encoder_inputs = keras.Input(shape=(28, 28, 1))
x = keras.layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
x = keras.layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
x = keras.layers.Flatten()(x)
x = keras.layers.Dense(16, activation="relu")(x)
z_mean = keras.layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = keras.layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

In [ ]:
latent_inputs = keras.Input(shape=(latent_dim,))
x = keras.layers.Dense(7 * 7 * 64, activation="relu")(latent_inputs)
x = keras.layers.Reshape((7, 7, 64))(x)
x = keras.layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
x = keras.layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
decoder_outputs = keras.layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(
            name="reconstruction_loss"
        )
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(X_train, epochs=30, batch_size=128)

In [ ]:
def auto_encode_example(example: numpy.ndarray) -> None:
  x_z_mean, x_z_log_var, x_z = vae.encoder(example[None, ...])
  decoded = vae.decoder(x_z).numpy().squeeze()
  plt.imshow(example.squeeze(), cmap="gray_r")
  plt.show()
  plt.imshow(decoded, cmap="gray_r")
  plt.show()


auto_encode_example(X_test[0])